In [ ]:
from google.colab import drive
import os

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Tạo thư mục chứa dữ liệu
os.makedirs('/content/dataset', exist_ok=True)

# 3. Copy và Giải nén ẩn (-q)
print("Đang copy dữ liệu Celeb-DF...")
!cp "/content/drive/MyDrive/DoAn_Nhom4/Celeb-DF-v2.zip" "/content/"

print("Đang giải nén dữ liệu...")
!unzip -q "/content/Celeb-DF-v2.zip" -d "/content/dataset/"

# 4. Xóa file nén để dọn rác ổ cứng
!rm "/content/Celeb-DF-v2.zip"
print("✅ Hoàn tất! Dữ liệu đã sẵn sàng tại /content/dataset/")

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install timm mediapipe opencv-python pandas scikit-learn onnx onnxscript onnxruntime gradio

In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

# Kiểm tra thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hệ thống đang chạy trên: {device}")

# Các phép biến đổi (Transforms)
swin_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

rppg_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Class Dataset Thực tế
class RealDeepfakeDataset(Dataset):
    def __init__(self, root_dir, visual_transform=None, bio_transform=None):
        self.visual_transform = visual_transform
        self.bio_transform = bio_transform
        self.image_paths = []
        self.labels = []

        real_dir = os.path.join(root_dir, 'real')
        fake_dir = os.path.join(root_dir, 'fake')

        # Đọc ảnh Real (Nhãn 0)
        if os.path.exists(real_dir):
            for file_name in os.listdir(real_dir):
                if file_name.endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(real_dir, file_name))
                    self.labels.append(0)

        # Đọc ảnh Fake (Nhãn 1)
        if os.path.exists(fake_dir):
            for file_name in os.listdir(fake_dir):
                if file_name.endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(fake_dir, file_name))
                    self.labels.append(1)

        print(f"Đã tìm thấy tổng cộng {len(self.image_paths)} hình ảnh.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        img_cv2 = cv2.imread(img_path)
        if img_cv2 is None:
            img_cv2 = np.zeros((224, 224, 3), dtype=np.uint8)

        img_rgb = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)

        visual_input = self.visual_transform(img_pil)
        rppg_input = self.bio_transform(img_pil) # Tạm thời dùng ảnh resize làm rPPG

        return visual_input, rppg_input, torch.tensor(label, dtype=torch.float32)

# Khởi tạo DataLoader
try:
    train_dataset = RealDeepfakeDataset('/content/dataset', swin_transform, rppg_transform)
    dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
    print("✅ Đã tạo DataLoader thực tế thành công!")
except Exception as e:
    print(f"❌ Có lỗi: {e}. Vui lòng kiểm tra lại cấu trúc thư mục.")

In [ ]:
import torch.nn as nn
import timm

class DeepfakeFusionModel(nn.Module):
    def __init__(self):
        super(DeepfakeFusionModel, self).__init__()
        # Nhánh Visual
        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)

        # Nhánh Sinh học
        self.rppg_net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        # Lớp Dung hợp (Fusion)
        self.classifier = nn.Sequential(
            nn.Linear(768 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, image_input, rppg_input):
        visual_features = self.swin(image_input)
        bio_features = torch.flatten(self.rppg_net(rppg_input), 1)

        combined_features = torch.cat((visual_features, bio_features), dim=1)
        output = self.classifier(combined_features)
        return output

model = DeepfakeFusionModel().to(device)
print("✅ Đã khởi tạo Mô hình Fusion thành công!")

In [ ]:
import torch.optim as optim

save_path = '/content/drive/MyDrive/DoAn_Nhom4/weights/'
os.makedirs(save_path, exist_ok=True)

criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

epochs = 30

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN ({epochs} EPOCHS) 🚀")
print("-" * 50)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, rppg_maps, labels) in enumerate(dataloader):
        images, rppg_maps, labels = images.to(device), rppg_maps.to(device), labels.float().to(device)

        optimizer.zero_grad()
        outputs = model(images, rppg_maps).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"🔄 Epoch [{epoch+1}/{epochs}] - Batch [{batch_idx}/{len(dataloader)}] - Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(dataloader)
    print(f"⭐ TỔNG KẾT EPOCH {epoch+1}: Trung bình Loss = {epoch_loss:.4f}")

    checkpoint_file = f"{save_path}fusion_model_epoch_{epoch+1}.pth"
    torch.save(model.state_dict(), checkpoint_file)
    print(f"✅ Đã lưu an toàn: {checkpoint_file}")
    print("-" * 50)

    torch.cuda.empty_cache()

print("🎉 HUẤN LUYỆN HOÀN TẤT!")